# 第11章 表格数据、星表与 HR 图

本 notebook 使用一个小型的 Gaia 风格教学数据，演示颜色指数、视差和绝对星等的基本计算。

## 学习目标

- 读取一个简单的恒星测光表。
- 计算颜色指数和绝对星等。
- 生成一个最小版本的 HR 图，并初步区分主序、巨星和白矮星区域。

In [ ]:
import csv
import math
from pathlib import Path

data_path = Path("../../data/small/stellar_photometry_demo.csv")
with data_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    row["bp_rp"] = float(row["bp_rp"])
    row["parallax_mas"] = float(row["parallax_mas"])
    row["g_mag"] = float(row["g_mag"])
    row["absolute_g_mag"] = row["g_mag"] - 10.0 + 5.0 * math.log10(row["parallax_mas"])

print("样本数量:", len(rows))
print("第一个样本:", rows[0])


## 视差质量与绝对星等误差

这份教学数据没有真实视差误差列，所以这里用一个假设的典型视差误差来演示正文公式。真实项目里应该使用星表提供的实际不确定度和质量标志。

In [ ]:
positive_parallax_rows = [row for row in rows if row['parallax_mas'] > 0]
print('positive parallax rows:', len(positive_parallax_rows), '/', len(rows))

assumed_sigma_parallax_mas = 0.2
assumed_sigma_g_mag = 0.02

for row in rows:
    sigma_from_parallax = (5.0 / math.log(10.0)) * (assumed_sigma_parallax_mas / row['parallax_mas'])
    row['absolute_g_mag_sigma_demo'] = math.sqrt(assumed_sigma_g_mag ** 2 + sigma_from_parallax ** 2)

sensitivity = sorted(
    (row['label'], row['parallax_mas'], row['absolute_g_mag_sigma_demo'])
    for row in rows
)
for label, parallax, sigma_m in sensitivity:
    print(f'{label:14s} parallax={parallax:6.1f} mas  sigma_M_demo={sigma_m:6.3f} mag')


## 查看几个关键派生量

In [ ]:
color_values = [row["bp_rp"] for row in rows]
absolute_magnitudes = [row["absolute_g_mag"] for row in rows]

print("最蓝恒星的 BP-RP:", min(color_values))
print("最亮样本的绝对星等:", round(min(absolute_magnitudes), 2))
print("最暗样本的绝对星等:", round(max(absolute_magnitudes), 2))


## HR 图上的厚度来自哪里

即使是真实主序带，也不会是一条无限细的线。测量误差、未解析双星、金属丰度、消光和样本选择都会让点云变厚。下面先按教学标签做一个最小分组摘要。

In [ ]:
class_summary = {}
for row in rows:
    class_summary.setdefault(row['stellar_class'], []).append(row)

for class_name, class_rows in class_summary.items():
    mean_color = sum(row['bp_rp'] for row in class_rows) / len(class_rows)
    mean_abs_mag = sum(row['absolute_g_mag'] for row in class_rows) / len(class_rows)
    print(f'{class_name:14s} n={len(class_rows)} mean BP-RP={mean_color:5.2f} mean M_G={mean_abs_mag:6.2f}')


## 如果已经安装 matplotlib，就绘制一个教学版 HR 图

In [ ]:
figure_path = Path("../../figures/generated/ch11_hr_diagram_demo.png")

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("matplotlib 未安装；已跳过 HR 图绘制。")
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    class_colors = {
        'main_sequence': 'tab:blue',
        'giant': 'tab:red',
        'white_dwarf': 'tab:green',
    }
    for class_name, class_rows in class_summary.items():
        ax.scatter(
            [row['bp_rp'] for row in class_rows],
            [row['absolute_g_mag'] for row in class_rows],
            color=class_colors.get(class_name, 'tab:gray'),
            label=class_name,
        )
    for row in rows:
        ax.annotate(row["label"], (row["bp_rp"], row["absolute_g_mag"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.set_title("Teaching HR Diagram Demo")
    ax.set_xlabel("BP - RP [mag]")
    ax.set_ylabel("Absolute G magnitude [mag]")
    ax.invert_yaxis()
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(figure_path, dpi=150)
    plt.close(fig)
    print("图像已保存到:", figure_path)


## 解释

在 HR 图里，横轴通常代表颜色或温度，纵轴代表光度或绝对星等。这个教学样本虽然很小，但已经足够用来说明主序星、巨星和白矮星在图上的相对位置差异。

## 练习建议

1. 给不同的恒星类别使用不同颜色。
2. 计算样本中蓝端和红端恒星的平均绝对星等。
3. 尝试解释为什么 HR 图中纵轴通常要反向显示。